# House Price Prediction with Categorical Encoding

In this notebook we will use a real-world housing dataset from Kaggle.

Goals of this notebook:

1. Explore a real dataset
2. Identify numerical and categorical variables
3. Handle missing data
4. Convert categorical variables into numerical form using encoding
5. Train a regression model

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

## Load the Dataset

In [ ]:
df = pd.read_csv("house-prices-advanced-regression-techniques/train.csv")

df.head()

## Dataset Overview

The dataset contains many features describing houses, including:

Numerical features:
- LotArea
- OverallQual
- YearBuilt
- GrLivArea

Categorical features:
- Neighborhood
- HouseStyle
- RoofStyle
- SaleCondition

The target variable is:

SalePrice

In [ ]:
df.info()

## Identify Data Types

In [ ]:
numerical_features = df.select_dtypes(include=["int64","float64"]).columns
categorical_features = df.select_dtypes(include=["object"]).columns

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

## Why Do We Need Encoding?

Machine learning models require numerical input.

However many variables in the dataset are categorical:

Example:

HouseStyle

- 1Story
- 2Story
- 1.5Fin

These must be converted into numerical representations before training the model.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["HouseStyle_encoded"] = le.fit_transform(df["HouseStyle"])

In [ ]:
df[["HouseStyle", "HouseStyle_encoded"]].head()

### One-Hot Encoding

One-Hot Encoding creates a separate binary column for each category.

Example:

| HouseStyle_1Story | HouseStyle_2Story | HouseStyle_1.5Fin |
|---|---|---|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

This avoids introducing artificial ordering.

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical_features)

In [ ]:
df_encoded.head()

## Preparing Data for Training

In [ ]:
X = df_encoded.drop("SalePrice", axis=1)
y = df_encoded["SalePrice"]

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train Linear Regression Model

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

In [ ]:
df_encoded.isnull().sum().sort_values(ascending=False).head(10)

## Addressing missing NA values
These are some missing values still remain in the dataset, we need to fix those too. Normally we can modify our previous code and correct this but, in this notebook I have rewrite the code for your reference 

In [ ]:
df[categorical_features] = df[categorical_features].fillna("None")

In [ ]:
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["GarageYrBlt"].median())
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical_features)

In [ ]:
df_encoded.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
X = df_encoded.drop("SalePrice", axis=1)
y = df_encoded["SalePrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

## Model Prediction

In [ ]:
predictions = model.predict(X_test)

## Evaluate Model

In [ ]:
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print("RMSE:", rmse)
print("R2:", r2)

## Visualization of Model Performance

In [ ]:
plt.scatter(y_test, predictions)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")

plt.title("Actual vs Predicted Prices")

plt.show()

## Save the Model

In [ ]:
import joblib

joblib.dump(model, "house_price_model.pkl")

## Load and Use the model

In [ ]:
model_load = joblib.load("house_price_model.pkl")

In [ ]:
new_house = pd.DataFrame({
    "area": [140],
    "bedrooms": [3],
    "bathrooms": [2],
    "distance_city": [5]
})

predicted_price = model_load.predict(new_house)

print("Predicted price:", predicted_price[0])

In [ ]:
for col in df_encoded.columns:
    print(col)